# 0921 16일차

## 1. train/test 분리가 없는 데이터

폴더 안에 클래스별 폴더만 있고 train/test 구분이 없는 데이터를, 통째로 불러온 뒤 배열 상태에서 나누는 방법

**필요한 이유**

1. brain은 `train/`, `test/`가 미리 나뉘어 있었지만 horse-or-human, rps는 클래스 폴더만 있음
2. 나누는 기준이 없으면 `flow_from_directory`를 두 번 부를 대상이 없음
3. 이미 쓰던 `train_test_split`을 그대로 쓸 수 있으므로 새 도구가 필요하지 않음

- 생성기로 **전체를 한 번에 배열로 만든 다음** 나누는 순서

### 1-1. 상위 폴더 지정

`flow_from_directory`는 넘겨준 경로의 **바로 아래 하위 폴더**를 클래스로 봄

1. brain은 `train/` 아래에 클래스 폴더가 있어서 `train/`을 넘겼음
2. horse-or-human은 상위 폴더 아래에 이미 클래스 폴더가 있으므로 상위 폴더를 넘김
3. 결과는 train/test 구분 없는 전체 한 덩어리

**주의) 정렬된 상태에서 잘림**

- 통째로 꺼내려면 `batch_size`가 전체 장수 이상이어야 함
- 작으면 0번 배치에 **앞에서부터 그 수만큼만** 담기는데, 섞지 않고 읽으면 클래스 순서대로 정렬돼 있으므로 뒤쪽 클래스가 통째로 빠짐
- 클래스가 하나만 남아도 에러가 나지 않고, 그 상태로 훈련하면 정확도 1.0이 나옴

### 1-2. 튜플 언패킹

여러 값이 묶인 튜플을 왼쪽에 변수를 나열해 한 번에 나눠 담는 문법

**필요한 이유**

1. 함수는 `return`을 한 번만 하므로, 값을 여러 개 돌려줄 때 튜플로 묶어서 내보냄
2. 묶인 채로 받으면 쓸 때마다 번호로 꺼내야 해서 번거로움

1. 왼쪽 변수 개수와 튜플 원소 개수가 같아야 함
2. 순서대로 담기므로 변수 이름은 자유지만 **순서를 바꾸면 값이 엇갈림**

- 배치 하나가 `(x, y)` 튜플이므로 여기에 언패킹을 씀
- `train_test_split`이 4개를 돌려주는 것도 같은 구조

### 1-3. class_mode

폴더 이름에서 만든 라벨을 **어떤 형태로 줄지** 정하는 옵션

1. `'binary'` : `(N,)`, 값은 `0.0` / `1.0`. 클래스 2개 전용
2. `'categorical'` : `(N, 클래스 수)`, 원핫
3. `'sparse'` : `(N,)`, 값은 `0` / `1` / `2` 같은 번호

- 라벨 번호를 매기는 규칙은 day15와 같음. 폴더 이름을 바꾸면 번호도 바뀜
- `'categorical'`은 원핫까지 만들어 주므로 `to_categorical`을 따로 부르지 않음
- 정형 데이터는 y가 컬럼으로 와서 직접 변환했지만, 이미지는 y의 출처가 폴더 이름이라 생성기가 만들 때 같이 처리함
- 클래스가 3개인데 `'binary'`를 쓰면 `0, 1, 2`가 되어 크기 순서가 생기므로 분류에 쓸 수 없음

### 1-4. 출력층과 loss의 짝

y의 형태 · 출력층 노드 수 · 활성화 함수 · loss는 하나의 묶음으로 맞춰야 함

| 분류 | class_mode | 출력층 | loss |
|---|---|---|---|
| 이진 | `'binary'` | `Dense(1, 'sigmoid')` | `binary_crossentropy` |
| 다중 | `'categorical'` | `Dense(클래스 수, 'softmax')` | `categorical_crossentropy` |
| 다중 | `'sparse'` | `Dense(클래스 수, 'softmax')` | `sparse_categorical_crossentropy` |

1. 출력층 노드 수는 두 방식 모두 **클래스 개수**만큼
2. 원핫이면 `categorical_crossentropy`, 번호면 `sparse_categorical_crossentropy`
3. 어긋나면 `Shapes (None, 1) and (None, 3) are incompatible` 형태로 걸림
   - 두 shape의 뒤쪽 숫자가 어긋났다는 뜻. `None`은 배치 크기 자리라 볼 필요 없음

### 1-5. 노드 1개에 softmax를 못 쓰는 이유

softmax는 출력 노드들의 **합이 1이 되도록 비율**을 계산하는 함수

```
노드 3개 : softmax([2, 1, 0]) -> [0.67, 0.24, 0.09]   합 1
노드 1개 : softmax([z])       -> 1.0                   항상
```

1. 노드가 하나면 나눌 상대가 없어 입력과 무관하게 항상 1.0이 나옴
2. 출력이 입력에 반응하지 않으므로 기울기가 0이 되어 가중치가 갱신되지 않음
3. epoch를 아무리 늘려도 첫 epoch 상태 그대로

- 이때 정확도는 학습 결과가 아니라 **한 쪽으로만 찍어서 맞은 비율**이 됨
- 이진분류에서 노드 1개를 쓸 때는 혼자서 0~1로 눌러 주는 sigmoid를 씀

### 1-6. 예측값을 라벨로 되돌리기

`predict`는 확률을 돌려주는데 `accuracy_score`는 라벨을 받으므로 변환이 필요함

1. 이진분류 : 확률 하나이므로 `round`로 0.5 기준으로 자름
2. 다중분류 : 확률이 클래스 수만큼이므로 `argmax`로 가장 큰 값의 **번호**를 꺼냄
3. y_test도 원핫이면 같이 번호로 되돌려야 비교됨

- 다중분류에 `round`를 쓰면 `[0.4, 0.35, 0.25]`처럼 애매할 때 전부 0이 되어 아무것도 고르지 않게 됨
- `argmax`는 항상 하나를 고름

### 1-7. target_size 정하기

이미지를 몇 픽셀로 줄여서 배열에 담을지 정하는 값

**정하는 기준**

1. 구분에 필요한 디테일이 남는 크기
   - 실루엣만 보면 되는 데이터는 작아도 되고, 미세한 질감을 봐야 하면 커야 함
2. 원본보다 키우지 않음
   - 없던 정보가 생기지 않고 계산량만 늘어남
3. 메모리
   - 가로세로를 늘리면 픽셀 수는 제곱으로 늘어남

- 작게 시작해 학습이 도는지 확인하고, 정확도가 부족할 때 키우는 순서
- 처음부터 크게 잡으면 오타 하나 확인하는 데도 시간이 오래 걸림

## 2. 타입별 인덱싱

`data[0]`이 무엇을 꺼내는지는 `data`에 **무엇이 담겨 있느냐**에 따라 달라짐

**필요한 이유**

1. 같은 대괄호라도 담긴 타입마다 꺼내지는 것이 다름
2. 타입이 어긋나도 에러 없이 엉뚱한 값이 나오는 경우가 있음
3. 에러가 나더라도 메시지가 원인에서 먼 곳을 가리킴

**타입별로 대괄호가 꺼내는 것**

1. 문자열 : 글자 하나
2. 생성기 : i번째 **배치**. x가 아니라 `(x, y)` 튜플
3. 튜플 : 그 안의 원소. 배치에서 x나 y를 꺼내는 자리
4. 넘파이 배열 : i번째 이미지

- `np.load`를 빠뜨리면 변수에 경로 문자열이 그대로 남는데, 문자열도 대괄호가 통하므로 알아채기 어려움
- 문자열은 0차원이라 `predict`에 넘기면 `tuple index out of range`가 남. 몇 장인지 세려고 `shape[0]`을 보는데 shape이 빈 튜플이라 꺼낼 자리가 없다는 뜻
- 메시지에 문자열이라는 말이 없어서 에러만 봐서는 원인을 알기 어려움
- 막히면 `type`과 `shape`부터 찍어볼 것. `shape`이 `()`로 나오면 배열이 아니거나 0차원

## 3. 이미지 한 장으로 예측

훈련까지 끝낸 모델을 불러와, 폴더가 아닌 **사진 한 장**을 넣어 결과를 보는 과정

**필요한 이유**

1. 실제로 쓸 때는 폴더째가 아니라 사진 한 장이 들어옴
2. 훈련은 오래 걸리므로 저장해 둔 모델을 다시 쓰는 편이 나음
3. 정답이 없는 이미지라 `evaluate`가 아니라 `predict`만 함

- `.keras` 파일에는 구조와 가중치가 같이 들어 있어 불러오기 한 번으로 끝남
- 예측만 할 거면 `compile`도 하지 않음

### 3-1. 파일 하나를 배열로 만들기

폴더 생성기를 쓰지 않고 파일 하나를 직접 읽어 4차원 배열로 만드는 순서

1. 이미지를 불러옴 — 이때 크기를 지정해 맞춤
2. 불러온 것은 이미지 객체라 모델에 바로 못 넣으므로 배열로 바꿈
3. 앞에 차원을 하나 늘림

- 모델은 언제나 `(장수, 세로, 가로, 채널)` 4차원을 받으므로, 한 장이어도 `(1, ...)` 형태여야 함
- 차원을 늘리지 않으면 3차원이라 장수 자리가 없어서 들어가지 않음

### 3-2. 주의) 학습 때와 같은 전처리

예측할 이미지는 **훈련 데이터와 똑같은 과정**을 거쳐야 함

1. 크기 — 훈련 때 지정한 크기와 같아야 함
2. 스케일 — 훈련 때 0~1로 줄였으면 예측할 이미지도 줄여야 함
3. 채널 — 흑백으로 훈련했으면 흑백으로 읽어야 함

- 크기가 다르면 에러가 나서 바로 알 수 있음
- 스케일이 다르면 **에러 없이 결과만 이상하게** 나옴. 모델이 한 번도 본 적 없는 범위의 값을 받기 때문
- 폴더 생성기는 스케일 조정을 자동으로 하지만, 파일 하나를 직접 읽으면 0~255 그대로이므로 따로 나눠야 함
- 나누는 곳은 저장할 때와 불러올 때 중 **한 곳만**. 양쪽에서 하면 두 번 나뉨

### 3-3. 예측값 읽기

출력층의 노드 수에 따라 같은 자리에서 꺼낸 값의 뜻이 달라짐

1. 노드 1개 : 값 하나가 **라벨 1일 확률**. 0.5보다 크면 1번 클래스
2. 노드 여러 개 : 클래스 수만큼의 확률. 가장 큰 자리의 번호가 답

- 어느 클래스가 0번인지는 폴더 이름의 알파벳 순서로 정해짐
- 노드 1개일 때만 `1 - 확률`이 반대쪽 확률이 됨. 노드가 여러 개면 각 자리를 직접 꺼내야 함
- 남녀나 개·고양이처럼 뒤바뀌어도 그럴듯해 보이는 경우가 있으므로, 답이 분명한 사진 한 장으로 확인하는 편이 빠름

## 4. flow로 증폭

이미 배열이 된 이미지에 변형을 주어 새 이미지를 만들어 내는 방법

**필요한 이유**

1. 폴더가 아니라 손에 든 배열을 늘려야 할 때가 있음
2. 사진 한 장만 있어도 여러 장을 만들어 낼 수 있음

- 변형 옵션은 `ImageDataGenerator`를 만들 때 지정하고, 실제로 만드는 것은 `flow`
- **주의)** 옵션은 대부분 비율인데 회전과 기울이기는 **각도**임. 비율로 착각해 작은 값을 주면 에러 없이 변형이 거의 일어나지 않음

### 4-1. flow와 flow_from_directory

이미지를 어디서 가져오느냐가 다름

1. `flow_from_directory` : 폴더 경로를 받아 파일을 읽음. 폴더 이름으로 y도 같이 만듦
2. `flow` : 이미 만들어 둔 배열을 받음. y는 따로 넘겨야 함

- 사진 한 장을 늘릴 때는 폴더가 없으므로 `flow`를 씀
- 넘기는 배열도 `(장수, 세로, 가로, 채널)` 4차원이어야 함

### 4-2. 주의) 꺼낼 때마다 새 이미지

`flow`가 돌려주는 것은 완성된 배열이 아니라, 꺼낼 때마다 새로 만들어 주는 이터레이터

1. 한 번 꺼낼 때마다 원본에 **다른 변형**이 적용된 이미지가 나옴
2. 원본이 한 장이어도 몇 장이든 꺼낼 수 있음
3. 반복문의 끝이 없으므로 몇 장을 만들지는 직접 정해야 함

- 같은 이미지를 두 번 보려면 변수에 담아 두어야 함. 다시 꺼내면 다른 이미지가 나옴
- 변형은 지정한 범위 안에서 무작위로 정해지므로 매번 다름